# Posterior predictive checks

Erik, 2026-11-18

A fitted model is also a generator of data. For each posterior draw, simulate a
replicated data set: new follow-up scores for the same students, from the
model with that draw's parameters. Together the replications form the posterior
predictive distribution: the data the model expects, given what it learned. A
posterior predictive check compares the observed data with the replications.
Where the observed data look unlike the replications, the model misses
something.

The check never proves a model right. It finds the features of the data a model
fails to reproduce, so the analyst can judge whether they matter for the
question. The notebook runs the checks on simulated students, where the model is
true by construction, and then on HSLS:09, using the draws saved by
`05-sampling`.

In [ ]:
import framework as fw
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats

fw.scaffold()
import bayes  # functions/bayes.py, on the path after scaffold()

sns.set_theme(style="whitegrid", palette="colorblind")

clean = fw.data_read("inputs.private.final.hsls_clean", dtype={"student_id": str})
hsls = bayes.analysis_sample(clean)
regression = bayes.model("regression")
data_spec = fw.data_read("inputs.public.raw.hsls_data_spec", col_types="c" * 14).fillna("")

hsls_draws = fw.load_model("hsls_regression_draws")
settings = fw.load_model("hsls_regression_settings")
assert settings["n"] == len(hsls), "The saved draws come from a different sample. Rerun 05-sampling."

# One generator for the simulated students and the replications
rng = np.random.default_rng(808)

## Test statistics

A test statistic summarizes a data set in one number. The check computes it for
the observed data and for each replication. The posterior predictive p-value is
the share of replications whose statistic is at least as large as the observed
one. Values near 0 or 1 mean the observed statistic is unusual under the model.
A value in between means the check found no discrepancy for that statistic,
which is weaker than saying the model is right about it. Statistics the model
fits directly, such as the mean, tend to give values near 0.5 because the same
data informed the fit.

In [ ]:
STATISTICS = {
    "Mean": np.mean,
    "SD": np.std,
    "Minimum": np.min,
    "Maximum": np.max,
    "Skewness": stats.skew,
    "5th percentile": lambda y: np.percentile(y, 5),
    "95th percentile": lambda y: np.percentile(y, 95),
    "Share above 2 SD (T-score 70)": lambda y: np.mean(y > 2),
}


def check(y, y_rep):
    rows = {}
    for name, statistic in STATISTICS.items():
        replicated = np.array([statistic(rep) for rep in y_rep])
        observed = statistic(y)
        rows[name] = {
            "Observed": observed,
            "Replicated, median": np.median(replicated),
            "Replicated, 5%": np.percentile(replicated, 5),
            "Replicated, 95%": np.percentile(replicated, 95),
            "p-value": np.mean(replicated >= observed),
        }
    return pd.DataFrame(rows).T


def overlay(y, y_rep, title, ax):
    for rep in y_rep[:50]:
        sns.kdeplot(rep, color="C0", alpha=0.15, linewidth=0.7, ax=ax)
    sns.kdeplot(y, color="black", linewidth=2, ax=ax)
    ax.set_xlabel("Follow-up math score (SD units)")
    ax.set_ylabel("Density")
    ax.set_title(title)

## Simulated data

The Stan program draws replications itself when `predict = 1`: its
`generated quantities` block draws `y_rep` for every student at every draw. The
cell fits 500 simulated students and takes the 4,000 by 500 array of
replications out of the fit.

In [ ]:
fake = bayes.simulate(500, 0.04, [0.7, 0.18], 0.66, rng)
fake_fit = bayes.sample(regression, bayes.stan_data(fake, bayes.WEAK, predict=True))
fake_rep = fake_fit.stan_variable("y_rep")
print(f"y_rep: {fake_rep.shape[0]:,} draws by {fake_rep.shape[1]} students")

fig, ax = plt.subplots(figsize=(7, 4))
overlay(fake["w2_math_z"].to_numpy(), fake_rep, "Simulated data (black) and 50 replications (blue)", ax)
plt.tight_layout()
# Every 4th draw: 1,000 replications spread evenly over the four chains
check(fake["w2_math_z"].to_numpy(), fake_rep[::4]).round(3)

When the model is true, the observed statistics sit inside the spread of the
replicated ones, and the p-values fall between 0 and 1 with no pattern. The
statistics use 1,000 of the 4,000 replications, every fourth draw. With 1,000
replications, a p-value near 0.5 has a simulation standard error of about
0.016, so differences in the second decimal are noise.

## HSLS:09

For 18,623 students, Stan's `y_rep` would hold 4,000 × 18,623 values, about
75 million. The replications are simple to draw outside Stan, with the same
formula as the `generated quantities` block:
$y^{\text{rep}}_i = \alpha + X_i \beta + \sigma \varepsilon_i$ with
$\varepsilon_i \sim \text{Normal}(0, 1)$. The cell draws one replication for
each of 400 posterior draws, chosen at random.

In [ ]:
x = hsls[bayes.PREDICTORS].to_numpy()
y = hsls[bayes.OUTCOME].to_numpy()
chosen = hsls_draws.sample(400, random_state=rng)
mu = chosen["alpha"].to_numpy()[:, None] + chosen[["beta[1]", "beta[2]"]].to_numpy() @ x.T
y_rep = mu + chosen["sigma"].to_numpy()[:, None] * rng.standard_normal(mu.shape)
print(f"y_rep: {y_rep.shape[0]} draws by {y_rep.shape[1]:,} students")

fig, ax = plt.subplots(figsize=(7, 4))
overlay(y, y_rep, "HSLS:09 follow-up scores (black) and 50 replications (blue)", ax)
plt.tight_layout()

In [ ]:
check(y, y_rep).round(3)

The model reproduces the mean and SD, which a linear regression with an
intercept fits by construction, and the upper part of the distribution: the 95th
percentile and the share above 2 SD (a T-score of 70). The tails are where it
fails:

- **Minimum and maximum.** Every replication has a lower minimum and a higher
  maximum than the observed scores. The observed minimum is about 0.8 SD (8
  points) above the median replicated minimum, and the observed maximum about
  0.6 SD (6 points) below the median replicated maximum. Normal errors have no
  bounds and the test's scores do. `03-simulating-the-data` anticipated this.
- **Lower tail.** The observed 5th percentile is below all 400 replicated
  ones, while the observed minimum is above all of them. More students score
  in the low 30s, as T-scores, than the model expects, and none score far
  below that.
- **Skewness.** The observed scores are slightly right-skewed, more than any
  replication.

The discrepancies are in the tails, so they matter most for questions about
the lowest and highest scorers. They say the normal distribution is an
approximation for this test score.

## Spread across base-year scores

The model assumes the same residual SD, $\sigma$, for every student. The check
splits students into ten groups by base-year score and compares the SD of the
follow-up score within each group with the same SD in the replications.

In [ ]:
deciles = pd.qcut(hsls["w1_math"], 10, labels=False)
observed_sd = pd.Series(y).groupby(deciles).std()
replicated_sd = np.array([pd.Series(rep).groupby(deciles).std().to_numpy() for rep in y_rep])
middle = hsls["w1_math"].groupby(deciles).median()

fig, ax = plt.subplots(figsize=(7, 4))
ax.fill_between(middle, np.percentile(replicated_sd, 5, axis=0), np.percentile(replicated_sd, 95, axis=0), alpha=0.3, label="Replications, central 90%")
ax.plot(middle, observed_sd, "o-", color="black", label="Observed")
ax.set_xlabel("Base-year math T-score, median of the decile")
ax.set_ylabel("SD of follow-up score within the decile (SD units)")
ax.set_title("Spread of follow-up scores by base-year decile")
ax.legend()
plt.tight_layout()
pd.DataFrame(
    {"Median base-year T-score": middle, "Observed SD": observed_sd, "Replicated SD, median": np.median(replicated_sd, axis=0)}
).round(3)

The observed SD in the middle deciles scatters around the replicated SD. At the
ends it does not. In the top decile, the observed SD is well below the
replicated one: the model overpredicts the spread among the students who scored
highest in the base year. A ceiling on the follow-up test is one explanation,
since it would bunch top scorers together; the check alone cannot confirm it. The replicated SD rises
in the outer deciles because those deciles span a wider range of base-year
scores, and the model passes that range on to the expected follow-up score. The
observed SD does not rise at the top. A model with a residual SD that varies
with the base-year score, or an outcome model with bounds, might fit better;
it would need the same check.

## Group means the model leaves out

The model has no term for race and ethnicity. If groups differ in follow-up
scores beyond what base-year math and SES predict, the replications will miss
their means. The check compares each group's observed mean follow-up score with
the replicated means for the same students.

In [ ]:
row = data_spec.set_index("name").loc["w1_race"]
race_labels = {float(v): label for v, label in (pair.split("=", 1) for pair in row["labels"].split(";"))}
race = hsls["w1_race"].map(race_labels).fillna("Missing")

observed_mean = pd.Series(y).groupby(race).mean()
replicated_mean = pd.DataFrame([pd.Series(rep).groupby(race).mean() for rep in y_rep])
groups = pd.DataFrame(
    {
        "Students": race.value_counts(),
        "Observed mean": observed_mean,
        "Replicated, 5%": replicated_mean.quantile(0.05),
        "Replicated, 95%": replicated_mean.quantile(0.95),
        "p-value": (replicated_mean >= observed_mean).mean(),
    }
).sort_values("Students", ascending=False)
groups.round(3)

In [ ]:
order = groups.index
fig, ax = plt.subplots(figsize=(8, 4.5))
positions = np.arange(len(order))
ax.hlines(positions, groups["Replicated, 5%"], groups["Replicated, 95%"], linewidth=6, alpha=0.4, label="Replications, central 90%")
ax.plot(groups["Observed mean"], positions, "o", color="black", label="Observed")
ax.set_yticks(positions)
ax.set_yticklabels([f"{name} (n = {n:,})" for name, n in zip(order, groups["Students"])], fontsize=8)
ax.set_xlabel("Mean follow-up math score (SD units)")
ax.set_title("Group means: observed and replicated")
ax.legend(fontsize=8)
plt.tight_layout()

Two groups fall outside the replicated band. The model underpredicts the mean
follow-up score of Asian students and overpredicts that of Black students,
given each group's own base-year math and SES. These are prediction errors for
group averages, not adjusted comparisons between groups: an estimate of how
groups differ at the same base-year math and SES needs a model with group
terms and a stated set of covariate values to compare at. A relationship that
is not linear could also produce errors like these, since the groups differ
in their base-year scores and SES. The other groups fall inside or at the edge
of their bands, and the small groups have wide bands. Whether to add race to
the model depends on the question. For describing how relative standing
persists, the misfit says one regression line averages over groups it does
not fit equally well. That is a substantive finding, not only a technical one.

## Predictions for one student

The posterior predictive distribution also answers a question the posterior
summary does not: what score to expect for one new student. For a student with a
base-year score of 50 and SES of 0, the cell compares two intervals:

- the 90% interval for the expected score, the average over all such students,
  which reflects uncertainty about the parameters only;
- the 90% predictive interval for one student's score, which adds the
  student-level variation $\sigma$.

In [ ]:
expected = hsls_draws["alpha"].to_numpy()
predicted = expected + hsls_draws["sigma"].to_numpy() * rng.standard_normal(len(expected))
pd.DataFrame(
    {
        "Expected score, average student": bayes.interval(expected, 0.9),
        "Score of one student": bayes.interval(predicted, 0.9),
    },
    index=["5%", "95%"],
).T.round(2)

The expected score is known to within about 0.01 SD, a tenth of a point. One
student's score is not: the predictive interval spans about 2.2 SD (22 points).
With this much data, almost all the uncertainty in a prediction for an
individual comes from the variation between students, not from the parameters.